<a href="https://colab.research.google.com/github/bubai-jkc/plant_disease_segmentation/blob/main/Dataset%203/Resnet101_BCE_BDoU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d fakhrealam9537/leaf-disease-segmentation-dataset

import zipfile, os, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm
import random

zip_ref = zipfile.ZipFile('/content/leaf-disease-segmentation-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

DATASET_PATH = "/content/aug_data/aug_data"
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
LR = 1e-4
NUM_EPOCHS = 120
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LeafDataset(Dataset):
    def __init__(self, root, augment=False):
        self.img_dir = os.path.join(root, "images")
        self.mask_dir = os.path.join(root, "masks")
        self.files = sorted(os.listdir(self.img_dir))
        self.augment = augment
        self.norm = transforms.Normalize([0.485,0.456,0.406],
                                         [0.229,0.224,0.225])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        img = Image.open(os.path.join(self.img_dir,name)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir,name.replace(".jpg",".png"))).convert("L")

        img = img.resize(IMG_SIZE)
        mask = mask.resize(IMG_SIZE, resample=Image.NEAREST)

        if self.augment:
            if np.random.rand()>0.5:
                img = transforms.functional.hflip(img)
                mask = transforms.functional.hflip(mask)
            if np.random.rand()>0.5:
                img = transforms.functional.vflip(img)
                mask = transforms.functional.vflip(mask)

        img = transforms.ToTensor()(img)
        img = self.norm(img)
        mask = (transforms.ToTensor()(mask)>0).float()

        return img, mask

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

full_dataset = LeafDataset(DATASET_PATH)
total = len(full_dataset)

train_size = int(0.7 * total)
val_size = int(0.15 * total)

split_path = "/content/data_split.pt"

if os.path.exists(split_path):
    indices = torch.load(split_path)
    print("Loaded fixed split")
else:
    indices = torch.randperm(total)
    torch.save(indices, split_path)
    print("Saved new split")

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size+val_size]
test_idx = indices[train_size+val_size:]

train_ds = torch.utils.data.Subset(LeafDataset(DATASET_PATH, True), train_idx)
val_ds   = torch.utils.data.Subset(LeafDataset(DATASET_PATH, False), val_idx)
test_ds  = torch.utils.data.Subset(LeafDataset(DATASET_PATH, False), test_idx)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class DoubleConv(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c,out_c,3,1,1,bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c,out_c,3,1,1,bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.net(x)

class Up(nn.Module):
    def __init__(self,in_c,skip_c,out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c,in_c//2,2,2)
        self.conv = DoubleConv(in_c//2+skip_c,out_c)

    def forward(self,x,skip):
        x = self.up(x)
        x = torch.cat([x,skip],1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        layers = list(base.children())

        self.init = nn.Sequential(*layers[:3])
        self.pool = layers[3]
        self.l1 = layers[4]
        self.l2 = layers[5]
        self.l3 = layers[6]
        self.l4 = layers[7]

        self.u1 = Up(2048,1024,512)
        self.u2 = Up(512,512,256)
        self.u3 = Up(256,256,128)
        self.u4 = Up(128,64,64)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(64,32,2,2),
            nn.Conv2d(32,1,1)
        )

    def forward(self,x):
        x0 = self.init(x)
        x1 = self.l1(self.pool(x0))
        x2 = self.l2(x1)
        x3 = self.l3(x2)
        x4 = self.l4(x3)

        d1 = self.u1(x4,x3)
        d2 = self.u2(d1,x2)
        d3 = self.u3(d2,x1)
        d4 = self.u4(d3,x0)

        return self.final(d4)


class BDoULoss(nn.Module):
    def __init__(self, alpha_adaptive=0.3, smooth=1e-6):
        super(BDoULoss, self).__init__()
        self.alpha = alpha_adaptive
        self.smooth = smooth

    def get_boundary(self, x):
        max_p = F.max_pool2d(x, 3, 1, 1)
        min_p = -F.max_pool2d(-x, 3, 1, 1)
        return max_p - min_p

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        P = self.get_boundary(inputs)
        G = self.get_boundary(targets)

        intersection = (P * G).sum(dim=(1,2,3))
        union = (P + G).sum(dim=(1,2,3)) - intersection

        loss = (union - intersection + self.smooth) / (union - self.alpha * intersection + self.smooth)
        return loss.mean()


class BceBDoU(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bdou = BDoULoss()
        self.lambda_bdou = .05

    def forward(self, pred, target):
        bce = self.bce(pred, target)
        bdou = self.bdou(pred, target)
        return bce + self.lambda_bdou * bdou


criterion = BceBDoU()


def mean_iou(pred,mask):
    pred = (torch.sigmoid(pred)>0.5).float()
    inter = (pred*mask).sum((1,2,3))
    union = pred.sum((1,2,3)) + mask.sum((1,2,3)) - inter
    return ((inter+1e-6)/(union+1e-6)).mean()


model = UNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=7,
    factor=0.5
)

best_iou = 0

train_losses, val_losses = [], []
train_ious, val_ious = [], []

for epoch in range(NUM_EPOCHS):
    criterion.lambda_bdou = min(0.05 + epoch * 0.004, 0.40)
    model.train()
    train_loss, train_iou = 0,0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for x,y in loop:
        x,y = x.to(DEVICE), y.to(DEVICE)

        out = model(x)
        loss = criterion(out,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_iou += mean_iou(out,y).item()

    model.eval()
    val_loss, val_iou = 0,0

    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)

            val_loss += criterion(out,y).item()
            val_iou += mean_iou(out,y).item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    train_iou /= len(train_loader)
    val_iou /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_ious.append(train_iou)
    val_ious.append(val_iou)

    print(f"\nEpoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f} | Val IoU = {val_iou:.4f}")
    scheduler.step(val_iou)
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_model.pth")
        print("------------>>> Best model saved <<<-------------\n")

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Train Loss vs Validation Loss")

plt.subplot(1,2,2)
plt.plot(train_ious, label="Train IoU")
plt.plot(val_ious, label="Val IoU")
plt.legend()
plt.title("Train IoU vs Validation IoU")

plt.savefig("training_curves.png")
plt.show()

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_iou, test_loss = 0,0

with torch.no_grad():
    for x,y in test_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)

        test_loss += criterion(out,y).item()
        test_iou += mean_iou(out,y).item()

avg_test_iou = test_iou / len(test_loader)
avg_test_loss = test_loss / len(test_loader)

print("\n\n---------------------------------")
print("---------------------------------")
print(f"\nFINAL TEST IoU:: {avg_test_iou:.4f}")
print(f"FINAL TEST LOSS:: {avg_test_loss:.4f}")
print("\n---------------------------------")
print("---------------------------------\n\n")

os.makedirs("all_test_results", exist_ok=True)
os.makedirs("sample_outputs", exist_ok=True)

idx, sample_count = 0,0
print("\n\nSome sample outputs:\n")
with torch.no_grad():
    for x,y in test_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        pred = (torch.sigmoid(out)>0.5).float()

        for i in range(x.size(0)):
            iou = mean_iou(out[i].unsqueeze(0), y[i].unsqueeze(0)).item()

            img = x[i].cpu().permute(1,2,0).numpy()
            img = img*[0.229,0.224,0.225] + [0.485,0.456,0.406]
            img = np.clip(img,0,1)

            gt = y[i].cpu().squeeze().numpy()
            pr = pred[i].cpu().squeeze().numpy()

            fig_all, ax_all = plt.subplots(1,3, figsize=(10,4))
            ax_all[0].imshow(img); ax_all[1].imshow(gt,cmap='gray'); ax_all[2].imshow(pr,cmap='gray')
            for a in ax_all: a.axis("off")
            plt.savefig(f"all_test_results/{idx}.png")
            plt.close()


            if sample_count < 20:
                error = np.abs(gt - pr)
                fig, ax = plt.subplots(1,4, figsize=(14,4))

                ax[0].imshow(img)
                ax[0].set_title("Image")

                ax[1].imshow(gt, cmap='gray')
                ax[1].set_title("Ground Truth")

                ax[2].imshow(pr, cmap='gray')
                ax[2].set_title(f"Prediction (IoU: {iou:.3f})")

                ax[3].imshow(error, cmap='hot')
                ax[3].set_title("Error Map")


                for a in ax: a.axis("off")

                plt.savefig(f"sample_outputs/{sample_count}.png")
                plt.show()

                sample_count += 1

            idx += 1

!zip -rq results_BCE_BDoU.zip all_test_results sample_outputs

from google.colab import files
files.download("results_BCE_BDoU.zip")
print("\n\n>>> Test images saved successfully using BEST model")

Dataset URL: https://www.kaggle.com/datasets/fakhrealam9537/leaf-disease-segmentation-dataset
License(s): CC0-1.0
100% 503M/503M [00:07<00:00, 71.4MB/s]

Saved new split
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 174MB/s]
Epoch 1/120: 100%|██████████| 258/258 [01:21<00:00,  3.18it/s]



Epoch 1: Train Loss = 0.4169 | Val Loss = 0.2141 | Val IoU = 0.6451
------------>>> Best model saved <<<-------------



Epoch 2/120: 100%|██████████| 258/258 [01:24<00:00,  3.04it/s]



Epoch 2: Train Loss = 0.2041 | Val Loss = 0.1653 | Val IoU = 0.6947
------------>>> Best model saved <<<-------------



...

Epoch 114/120: 100%|██████████| 258/258 [01:26<00:00,  2.99it/s]



Epoch 114: Train Loss = 0.1498 | Val Loss = 0.2109 | Val IoU = 0.9116


Epoch 115/120: 100%|██████████| 258/258 [01:25<00:00,  3.01it/s]



Epoch 115: Train Loss = 0.1493 | Val Loss = 0.2103 | Val IoU = 0.9103


Epoch 116/120: 100%|██████████| 258/258 [01:26<00:00,  2.99it/s]
